# 06 - Model Evaluation

Since we're predicting a category (default / no default), **accuracy alone is not enough** — guessing "no default" every time already gets 88% accuracy without being useful. We check several scorecards together:

- **Accuracy** — % of all guesses correct. Easy to understand, but misleading here.
- **Precision** — of the borrowers FLAGGED as risky, how many actually defaulted?
- **Recall** — of the borrowers who ACTUALLY defaulted, how many did the model catch? Usually what a lender cares about most.
- **F1** — balances precision and recall.
- **ROC-AUC** — how good the model is at ranking people from safest to riskiest. 0.5 = random guessing, 1.0 = perfect.

**Input:** `../04-feature-engineering/data/X_test.csv`, `y_test.csv`, `../05-model-training/models/*.joblib`
**Output:** `output/model_comparison.csv`


In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix)

DATA_DIR = "../04-feature-engineering/data"
MODEL_DIR = "../05-model-training/models"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv")).iloc[:, 0]

model_files = {
    "Logistic Regression":           "logistic_regression.joblib",
    "Ridge-penalised Logistic (L2)": "ridge_logistic_l2.joblib",
    "Lasso-penalised Logistic (L1)": "lasso_logistic_l1.joblib",
    "Decision Tree":                 "decision_tree.joblib",
    "Random Forest":                 "random_forest.joblib",
    "Gradient Boosting":             "gradient_boosting.joblib",
}
fitted = {name: joblib.load(os.path.join(MODEL_DIR, f)) for name, f in model_files.items()}
print(f"Loaded {len(fitted)} trained models and {len(X_test):,} test rows")


Loaded 6 trained models and 76,605 test rows


In [2]:
results = []
for name, pipe in fitted.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall":    recall_score(y_test, pred),
        "F1":        f1_score(y_test, pred),
        "ROC_AUC":   roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
print("Scorecard for every model (higher is better everywhere):\n")
print(results_df.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

majority_acc = accuracy_score(y_test, np.zeros(len(y_test)))
print(f"\nFor comparison - 'always guess no default': Accuracy = {majority_acc:.4f}, Recall = 0.0000")

best_name = results_df.iloc[0]["Model"]
best_pred = fitted[best_name].predict(X_test)
cm = confusion_matrix(y_test, best_pred)
print(f"\nConfusion matrix for the top model by ROC-AUC ({best_name}):")
print(f"                     Predicted no default   Predicted default")
print(f"   Actually no default   {cm[0,0]:>10,}          {cm[0,1]:>10,}")
print(f"   Actually defaulted    {cm[1,0]:>10,}          {cm[1,1]:>10,}")
print(f"\n   Caught {cm[1,1]:,} of {cm[1,0]+cm[1,1]:,} actual defaulters "
      f"({cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%).")

results_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)
print("\nSaved: output/model_comparison.csv")


Scorecard for every model (higher is better everywhere):

                        Model  Accuracy  Precision   Recall       F1  ROC_AUC
            Gradient Boosting    0.8862     0.6282   0.0496   0.0919   0.7571
          Logistic Regression    0.6773     0.2199   0.6981   0.3344   0.7530
Ridge-penalised Logistic (L2)    0.6773     0.2199   0.6981   0.3344   0.7530
Lasso-penalised Logistic (L1)    0.6772     0.2198   0.6981   0.3343   0.7530
                Random Forest    0.7829     0.2735   0.5248   0.3596   0.7510
                Decision Tree    0.6557     0.2038   0.6759   0.3132   0.7230

For comparison - 'always guess no default': Accuracy = 0.8839, Recall = 0.0000



Confusion matrix for the top model by ROC-AUC (Gradient Boosting):
                     Predicted no default   Predicted default
   Actually no default       67,448                 261
   Actually defaulted         8,455                 441

   Caught 441 of 8,896 actual defaulters (5.0%).

Saved: output/model_comparison.csv


## How to read this — the most important lesson in the whole project

- **Gradient Boosting has the best Accuracy (~88.6%) and best ROC-AUC (~0.757)** — looks like the clear winner at a glance.
- **But its Recall is only ~0.05** — it catches just 5% of actual defaulters, barely better than not modelling at all. Its high accuracy is basically the "always guess no default" trick in disguise.
- **Logistic Regression has lower accuracy (~67.7%) but catches ~70% of actual defaulters** (Recall ≈ 0.698). It raises more false alarms (Precision ≈ 0.22), but for a lender trying to avoid losses, catching real defaulters usually matters more than a high accuracy score.
- **Lesson:** never trust one single number, especially accuracy, when the classes are this unbalanced (88% vs 12%). Always look at Recall and Precision together with the confusion matrix.

In [3]:
logreg = fitted["Logistic Regression"]
feature_names = logreg.named_steps["prep"].get_feature_names_out()
coefs = logreg.named_steps["model"].coef_[0]

coef_df = (pd.DataFrame({"feature": feature_names, "weight": coefs})
           .assign(size=lambda d: d.weight.abs())
           .sort_values("size", ascending=False)
           .drop(columns="size")
           .reset_index(drop=True))

print("The 10 things that matter most to the Logistic Regression model:\n")
print(coef_df.head(10).to_string(index=False, float_format=lambda v: f"{v:+8.4f}"))
print("\nNegative weight -> pushes prediction toward 'no default'")
print("Positive weight -> pushes prediction toward 'default'")


The 10 things that matter most to the Logistic Regression model:

                          feature   weight
                         num__Age  -0.5817
                num__InterestRate  +0.4572
   cat__EmploymentType_Unemployed  +0.4403
              num__MonthsEmployed  -0.3359
                      num__Income  -0.3098
                  num__LoanAmount  +0.2851
    cat__EmploymentType_Part-time  +0.2611
cat__EmploymentType_Self-employed  +0.2532
       cat__MaritalStatus_Married  -0.1901
            cat__LoanPurpose_Home  -0.1803

Negative weight -> pushes prediction toward 'no default'
Positive weight -> pushes prediction toward 'default'


**How to read this:**

- `Age` has the strongest weight, and it's negative — older borrowers are pushed toward "no default".
- `InterestRate` is strongly positive — a higher interest rate pushes toward "default", matching how lenders already price risk.
- Being unemployed is also strongly positive.
- Every sign matches common sense, a good check that the model learned something sensible.
- **Important:** these weights show association, not proof of cause. We cannot say "being unemployed CAUSES default" from this alone.

In [4]:
print(f"Total borrowers (train+test) : {len(X_test)*100//30:,} (approx)")
print(f"Best model (ROC-AUC) : {best_name}  ({results_df.iloc[0]['ROC_AUC']:.3f})")
print(f"Best model (Recall)  : "
      f"{results_df.sort_values('Recall', ascending=False).iloc[0]['Model']}  "
      f"({results_df['Recall'].max():.3f})")

print("""
THINGS TO MENTION IN A WRITE-UP
  1. This shows ASSOCIATION, not proof of what CAUSES default.
  2. Accuracy alone is misleading here - only 11.6% of borrowers defaulted.
  3. Results come from one dataset; a different bank or time period could differ.
  4. We used default model settings - no fine-tuning was done.
  5. We checked for "cheating" columns and found none (see Module 3).
""")


Total borrowers (train+test) : 255,350 (approx)
Best model (ROC-AUC) : Gradient Boosting  (0.757)
Best model (Recall)  : Logistic Regression  (0.698)

THINGS TO MENTION IN A WRITE-UP
  1. This shows ASSOCIATION, not proof of what CAUSES default.
  2. Accuracy alone is misleading here - only 11.6% of borrowers defaulted.
  3. Results come from one dataset; a different bank or time period could differ.
  4. We used default model settings - no fine-tuning was done.
  5. We checked for "cheating" columns and found none (see Module 3).



## Summary

The honest takeaway is not "one model is best" — it's that **different models make different trade-offs**. Gradient Boosting looks best on paper (accuracy, ROC-AUC) but is nearly useless at catching defaulters. Logistic Regression looks worse on paper but catches far more of the borrowers a lender actually cares about. Which one is "better" depends on what mistake is more costly: missing a real defaulter, or wrongly flagging a safe borrower. That trade-off *is* the finding.

### Project pipeline recap
1. **01-data-collection** — loaded 255,347 loan applications, first look.
2. **02-exploratory-analysis** — averages, correlation heatmap, charts.
3. **03-data-preprocessing** — checked duplicates/leakage, dropped the ID column.
4. **04-feature-engineering** — encoded ordinal columns, split into train/test (70/30, stratified).
5. **05-model-training** — trained 6 models with a shared preprocessing pipeline.
6. **06-model-evaluation** — scored every model honestly on unseen data, and inspected what the simplest model learned.
